# Losses: Proper Scoring, Evidential, and Differentiable Ranking (C03 Topic 11)

**Citations:** Brier (1950); Gneiting & Raftery (2007 CRPS); Xia et al. (2008 ListMLE); Sensoy et al. (2018 Evidential DL); Blondel et al. (2020 Soft Ranks); Seitzer et al. (2022 Beta-NLL).

### Purpose & Topic Bridges
Point predictions or uncalibrated softmax logits lack reliable uncertainty quantification in high-stakes domains. Proper scoring rules (CRPS, Brier Score) guarantee honest probabilistic forecasting where the true data distribution uniquely minimizes expected loss. Evidential Deep Learning parameterizes Dirichlet priors over multinomial distributions to capture epistemic vs. aleatoric uncertainty. In heteroscedastic regression, standard Gaussian NLL suffers from *gradient vanishing on noisy outliers*, which $\beta$-NLL cures via variance discounting. In ranking systems, ListMLE and Soft-Spearman enable end-to-end differentiable ranking.

**Decision Boundary:** Use CRPS / Brier for calibrated probabilistic forecasting. Use $\beta$-NLL ($\beta \in [0.5, 1.0]$) rather than standard NLL when training heteroscedastic neural networks to prevent outlier gradient starvation. Use ListMLE / Soft-Ranks when optimizing permutation-based retrieval metrics rather than top-1 cross-entropy.

## 2. Mathematical Foundations & Governing Equations

1. **Continuous Ranked Probability Score (CRPS):** For $\mathcal{N}(\mu, \sigma^2)$, $\text{CRPS} = \sigma [z(2\Phi(z) - 1) + 2\phi(z) - 1/\sqrt{\pi}]$ with $z = (y - \mu)/\sigma$.
2. **Brier Score:** $\text{BS}(p, y) = \sum_{k=1}^K (p_k - y_k)^2$ as a strictly proper quadratic scoring rule for categorical forecasts on $\Delta_{K-1}$.
3. **$\beta$-NLL:** $\mathcal{L}_{\beta\text{-NLL}} = (\hat{\sigma}^2)^\beta [\frac{1}{2}\log \sigma^2 + \frac{(y-\mu)^2}{2\sigma^2}]$, preventing $\nabla_\mu \mathcal{L} \to 0$ when $\sigma^2 \to \infty$.
4. **Evidential Deep Learning (EDL MSE):** $\mathcal{L}_{\text{EDL}} = \sum_{k=1}^K [(y_k - \hat{p}_k)^2 + \frac{\alpha_k(S - \alpha_k)}{S^2(S+1)}]$ with Dirichlet concentration $\alpha_k = e_k + 1, S = \sum \alpha_k$.
5. **ListMLE Ranking Loss:** $\mathcal{L}_{\text{ListMLE}} = -\sum_{i=1}^K [s_{\pi(i)} - \log \sum_{j=i}^K \exp(s_{\pi(j)})]$ under Plackett-Luce permutation likelihood.
6. **Soft-Spearman Ranks:** $R_i = 0.5 + \sum_{j=1}^K \sigma((s_i - s_j)/\tau)$, providing continuous, smooth differentiable ranking statistics.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/03_building_blocks_and_losses/11_losses_proper_scoring_evidential_and_differentiable_ranking.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class ProperScoringAndRankingModule(nn.Module):
    def __init__(self, in_dim=4, out_dim=4):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x): return self.linear(x)

    def crps_gaussian(self, y, mu, sigma, eps=1e-6):
        sig = sigma.clamp(min=eps); z = (y - mu) / sig
        phi = torch.exp(-0.5 * z.pow(2)) / math.sqrt(2.0 * math.pi)
        Phi = 0.5 * (1.0 + torch.erf(z / math.sqrt(2.0)))
        return (sig * (z * (2.0 * Phi - 1.0) + 2.0 * phi - 1.0 / math.sqrt(math.pi))).mean()

    def brier_score(self, p, y_onehot): return torch.mean(torch.sum((p - y_onehot).pow(2), dim=-1))

    def beta_nll(self, y, mu, var, beta=0.5, eps=1e-6):
        v = var.clamp(min=eps); loss = 0.5 * (torch.log(v) + (y - mu).pow(2) / v)
        return torch.mean((v.detach()).pow(beta) * loss)

    def edl_mse(self, alpha, y_onehot):
        S = torch.sum(alpha, dim=-1, keepdim=True); p_hat = alpha / S
        var_p = (alpha * (S - alpha)) / (S.pow(2) * (S + 1.0))
        return torch.sum((y_onehot - p_hat).pow(2) + var_p, dim=-1).mean()

    def listmle(self, scores, perm):
        B, K = scores.shape; total = 0.0
        for b in range(B):
            s_ord = scores[b][perm[b]]
            for i in range(K): total = total - (s_ord[i] - torch.logsumexp(s_ord[i:], dim=0))
        return total / float(B * K)

    def soft_ranks(self, scores, tau=0.5):
        diff = (scores.unsqueeze(2) - scores.unsqueeze(1)) / tau
        return 0.5 + torch.sum(torch.sigmoid(diff), dim=-1)

model = ProperScoringAndRankingModule().to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].t()); model.linear.bias.copy_(weights['bias'])
print('ProperScoringAndRankingModule initialized successfully.')

In [ ]:
err_crps = (model.crps_gaussian(weights['crps_y'], weights['crps_mu'], weights['crps_sigma']) - weights['expected_crps'][0]).abs().item()
assert err_crps < 1e-5, f'CRPS parity error: {err_crps:.2e}'

err_brier = (model.brier_score(weights['brier_p'], weights['brier_y']) - weights['expected_brier'][0]).abs().item()
assert err_brier < 1e-5, f'Brier parity error: {err_brier:.2e}'

err_bnll = (model.beta_nll(weights['beta_nll_y'], weights['beta_nll_mu'], weights['beta_nll_var'], beta=0.5) - weights['expected_beta_nll'][0]).abs().item()
assert err_bnll < 1e-5, f'Beta-NLL parity error: {err_bnll:.2e}'

err_edl = (model.edl_mse(weights['edl_alpha'], weights['edl_y']) - weights['expected_edl'][0]).abs().item()
assert err_edl < 1e-5, f'EDL MSE parity error: {err_edl:.2e}'

err_listmle = (model.listmle(weights['listmle_scores'], weights['listmle_perm']) - weights['expected_listmle'][0]).abs().item()
assert err_listmle < 1e-5, f'ListMLE parity error: {err_listmle:.2e}'

err_soft = (model.soft_ranks(weights['soft_scores'], tau=0.5) - weights['expected_soft_ranks']).abs().max().item()
assert err_soft < 1e-5, f'Soft ranks parity error: {err_soft:.2e}'

err_anchor = (model(test_in) - test_out).abs().max().item()
assert err_anchor < 1e-5, f'Anchor parity error: {err_anchor:.2e}'

print(f'All 7 invariants verified! CRPS: {err_crps:.2e}, Brier: {err_brier:.2e}, BetaNLL: {err_bnll:.2e}, EDL: {err_edl:.2e}, ListMLE: {err_listmle:.2e}, SoftRanks: {err_soft:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'Gaussian CRPS': lambda: model.crps_gaussian(weights['crps_y'], weights['crps_mu'], weights['crps_sigma']),
    'Brier Score': lambda: model.brier_score(weights['brier_p'], weights['brier_y']),
    'Beta-NLL': lambda: model.beta_nll(weights['beta_nll_y'], weights['beta_nll_mu'], weights['beta_nll_var']),
    'EDL MSE': lambda: model.edl_mse(weights['edl_alpha'], weights['edl_y']),
    'ListMLE': lambda: model.listmle(weights['listmle_scores'], weights['listmle_perm']),
    'Soft Ranks': lambda: model.soft_ranks(weights['soft_scores']),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Active Failure Mode: Heteroscedastic Gaussian NLL Gradient Vanishing Collapse
# When an outlier has large error (y - mu = 5.0) and high uncertainty (var = 100.0),
# standard NLL (beta=0) gradient w.r.t. mu vanishes: dL/dmu = -(y - mu)/var -> 0!
mu_vanish = torch.tensor([0.0], requires_grad=True, device=device)
var_high = torch.tensor([100.0], device=device); y_target = torch.tensor([5.0], device=device)
loss_std = 0.5 * (torch.log(var_high) + (y_target - mu_vanish).pow(2) / var_high)
grad_std = torch.autograd.grad(loss_std, mu_vanish)[0].item()
print(f'[FAILURE DEMONSTRATED] Standard NLL gradient: {grad_std:.4f} (vanished toward 0 on noisy outlier!)')
assert abs(grad_std) < 0.1, f'Expected vanishing gradient, got {grad_std}'

# Beta-NLL (beta=1.0) variance discounting retains full restorative gradient force:
mu_active = torch.tensor([0.0], requires_grad=True, device=device)
loss_beta = (var_high.detach()).pow(1.0) * (0.5 * (torch.log(var_high) + (y_target - mu_active).pow(2) / var_high))
grad_beta = torch.autograd.grad(loss_beta, mu_active)[0].item()
print(f'[STABILIZED] Beta-NLL (beta=1.0) gradient: {grad_beta:.4f} (100x stronger restorative force!)')
assert abs(grad_beta) > 4.0, f'Expected active gradient >= 4.0, got {grad_beta}'
assert torch.allclose(model(test_in), test_out, atol=1e-5), 'Linear anchor mismatch'
print('[CORRECT] Proper scoring, evidential, and ranking loss mechanisms verified successfully.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Strict Properness Prevents Hedging:** CRPS and Brier score enforce honest predictive distributions where hedging yields strictly sub-optimal loss.
2. **Curing NLL Outlier Starvation:** Standard heteroscedastic NLL ignores difficult outlier samples by inflating variance. Setting $\beta \in [0.5, 1.0]$ in $\beta$-NLL guarantees consistent restorative gradient dynamics.
3. **From Categorical to Ranking:** ListMLE and Soft-Spearman ranks provide continuous relaxations of combinatorial permutation operators, enabling direct optimization of retrieval ranking metrics.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 10_losses_optimal_transport_wasserstein_sinkhorn_and_mmd](file:///c:/Users/Liam/Code/AI-Examples/notebooks/03_building_blocks_and_losses/10_losses_optimal_transport_wasserstein_sinkhorn_and_mmd.ipynb)